# Parsing de Documentos — Parte 1: Como os Dados Chegam

**Cenário:** você vai construir um pipeline de IA — RAG, fine-tuning, análise de contratos.  
Antes de qualquer modelo brilhar, você precisa de **dados limpos**.

| Tipo de dado | Exemplo concreto |
|---|---|
| HTML / texto com marcações | Páginas web, exports de sistemas, e-mails em HTML |
| PDF nativo (digital) | Contratos gerados por Word, políticas internas |
| PDF com tabelas | Relatórios financeiros, balanços, folha de pagamento |
| PDF complexo (misto) | Documentos com texto, tabelas, imagens, cabeçalhos hierárquicos |
| Documento escaneado | Notas fiscais fotografadas, documentos antigos digitalizados |

> **Parsing é a ponte entre o documento bruto e o dado utilizável.**  
> Um modelo é tão bom quanto os dados que recebe. *Garbage in, garbage out.*

## Instalação de dependências

In [ ]:
!pip install reportlab Pillow PyMuPDF -q

---
## 1.1 — HTML / Texto com Marcações

Uma página web, um e-mail exportado, um sistema legado — o conteúdo chega **embrulhado em tags**.  
O texto que importa está lá, mas escondido no meio de centenas de marcações.

In [ ]:
html_bruto = """
<!DOCTYPE html>
<html>
  <head>
    <meta charset="UTF-8">
    <title>Contrato de Prestação de Serviços</title>
    <style>
      body { font-family: Arial; color: #333; margin: 40px; }
      .destaque { background-color: #ffffcc; font-weight: bold; }
      table { border-collapse: collapse; width: 100%; }
      td, th { border: 1px solid #ccc; padding: 8px; }
    </style>
  </head>
  <body>
    <h1>CONTRATO Nº&nbsp;2024/001</h1>
    <p>
      Entre as partes: <strong>Empresa XYZ Ltda</strong>
      (CONTRATANTE) e <span class="destaque">João da Silva</span> (CONTRATADO).
    </p>
    <!-- Cláusula revisada em 15/03/2024 pelo jurídico -->
    <p>O presente contrato tem as seguintes condições:</p>
    <ul>
      <li>Valor total: <strong>R$&nbsp;5.000,00</strong></li>
      <li>Prazo de execução: <em>30 dias corridos</em></li>
      <li>Forma de pagamento: <em>50% na assinatura, 50% na entrega</em></li>
    </ul>
    <h2>Detalhamento de Serviços</h2>
    <table>
      <tr><th>Item</th><th>Descrição</th><th>Valor</th></tr>
      <tr><td>1</td><td>Desenvolvimento de sistema</td><td>R$ 3.000,00</td></tr>
      <tr><td>2</td><td>Consultoria técnica</td><td>R$ 2.000,00</td></tr>
    </table>
    <p><small>Gerado automaticamente em 28/03/2026 às 10:42:15</small></p>
  </body>
</html>
"""

print("=" * 60)
print("HTML BRUTO — o que chega do sistema")
print("=" * 60)
print(html_bruto)
print(f"\nTotal de caracteres: {len(html_bruto)}")
print(f"Desses, quanto é conteúdo real? Difícil saber sem parsing.")

Se você simplesmente jogar esse HTML num modelo de linguagem ou num campo de busca, ele vai processar **tags, atributos, comentários e CSS junto com o conteúdo**. O resultado é ruído.

---
## 1.2 — PDF Nativo (Digital)

PDFs gerados por Word, LibreOffice ou sistemas internos **parecem** texto — você consegue selecionar com o mouse.  
Mas por dentro são binários. Veja o que o computador vê quando tenta ler diretamente.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
import io

# Criar um PDF nativo simples em memória
buffer = io.BytesIO()
c = canvas.Canvas(buffer, pagesize=A4)
largura, altura = A4

c.setFont("Helvetica-Bold", 14)
c.drawString(72, altura - 72, "POLÍTICA INTERNA — USO DE SISTEMAS")

c.setFont("Helvetica", 11)
c.drawString(72, altura - 110, "Versão: 3.2   |   Data: 28/03/2026   |   Classificação: Interno")

c.setFont("Helvetica", 10)
linhas = [
    "1. OBJETIVO",
    "Esta política define as diretrizes para uso dos sistemas corporativos.",
    "",
    "2. ABRANGÊNCIA",
    "Aplica-se a todos os colaboradores com acesso aos sistemas da empresa.",
    "",
    "3. RESPONSABILIDADES",
    "3.1 O usuário é responsável pela confidencialidade de suas credenciais.",
    "3.2 É proibido o compartilhamento de senhas entre colaboradores.",
    "3.3 Acessos suspeitos devem ser reportados ao time de segurança.",
]

y = altura - 150
for linha in linhas:
    c.drawString(72, y, linha)
    y -= 18

c.save()
pdf_bytes = buffer.getvalue()

# --- O que o computador vê se tentar ler diretamente ---
print("=" * 60)
print("PDF NATIVO — primeiros 600 bytes brutos (binário)")
print("=" * 60)
print(pdf_bytes[:600])
print(f"\nTamanho total do arquivo: {len(pdf_bytes):,} bytes")

In [ ]:
# Tentativa ingênua: decodificar como texto comum
print("=" * 60)
print("Tentativa ingênua: pdf_bytes.decode('latin-1')")
print("=" * 60)
texto_ingenuo = pdf_bytes.decode('latin-1', errors='replace')
print(texto_ingenuo[:1000])
print("\n... [o texto real está lá em algum lugar, mas junto com centenas de metadados binários]")

O texto até aparece — mas misturado com cabeçalhos binários, operadores gráficos do PDF e metadados de fonte.  
Sem um parser adequado, você não consegue extrair nem saber a **ordem correta** das linhas.

---
## 1.3 — PDF com Tabelas

Relatórios financeiros, balanços, folhas de pagamento — dados estruturados presos em grade.  
O desafio: a tabela no PDF **não existe como estrutura de dados**, apenas como posições absolutas de texto na página.

In [ ]:
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
import io

buffer_tabela = io.BytesIO()
doc = SimpleDocTemplate(buffer_tabela, pagesize=A4)
styles = getSampleStyleSheet()

dados_tabela = [
    ["Funcionário",       "Cargo",            "Salário Base",  "Desconto INSS", "Líquido"],
    ["Ana Paula Costa",   "Analista Sênior",   "R$ 8.500,00",   "R$ 935,00",     "R$ 7.565,00"],
    ["Bruno Martins",     "Desenvolvedor Jr",  "R$ 4.200,00",   "R$ 462,00",     "R$ 3.738,00"],
    ["Carla Rodrigues",   "Gestora de Produto","R$ 12.000,00",  "R$ 1.320,00",   "R$ 10.680,00"],
    ["Diego Ferreira",    "Estagiário",        "R$ 1.800,00",   "R$ 0,00",       "R$ 1.800,00"],
    ["TOTAL",             "",                  "R$ 26.500,00",  "R$ 2.717,00",   "R$ 23.783,00"],
]

tabela = Table(dados_tabela, colWidths=[130, 120, 90, 90, 90])
tabela.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2c3e50')),
    ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
    ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE',   (0, 0), (-1, -1), 9),
    ('ROWBACKGROUNDS', (0, 1), (-1, -2), [colors.white, colors.HexColor('#ecf0f1')]),
    ('BACKGROUND', (0, -1), (-1, -1), colors.HexColor('#bdc3c7')),
    ('FONTNAME',   (0, -1), (-1, -1), 'Helvetica-Bold'),
    ('GRID',       (0, 0),  (-1, -1), 0.5, colors.grey),
    ('ALIGN',      (2, 0),  (-1, -1), 'RIGHT'),
    ('TOPPADDING', (0, 0),  (-1, -1), 6),
    ('BOTTOMPADDING', (0,0),(-1,-1), 6),
]))

titulo = Paragraph("<b>FOLHA DE PAGAMENTO — MARÇO/2026</b>", styles['Title'])
doc.build([titulo, tabela])
pdf_tabela_bytes = buffer_tabela.getvalue()

print("=" * 60)
print("PDF COM TABELA — como o texto é armazenado internamente")
print("=" * 60)

# Mostrar trecho do stream interno do PDF (onde ficam as coordenadas)
conteudo = pdf_tabela_bytes.decode('latin-1', errors='replace')
# Pegar só a parte do stream de conteúdo
inicio_stream = conteudo.find('stream\n')
if inicio_stream > 0:
    print(conteudo[inicio_stream:inicio_stream + 800])
print("\n[Cada 'Td', 'Tf', 'BT', 'ET' é um comando gráfico do PDF]")
print("[As colunas NÃO existem como estrutura — só como coordenadas X,Y na página]")

O problema central de tabelas em PDF:
- Célula `(linha=2, coluna=3)` não existe no arquivo
- O que existe é: *"texto 'R$ 8.500,00' na posição x=420, y=680"*
- Para reconstruir a tabela, você precisa inferir a grade a partir das posições

---
## 1.4 — Documento Escaneado

Notas fiscais fotografadas, contratos antigos, documentos de cartório.  
**Para o computador, isso é só uma imagem** — uma matriz de pixels. Não há texto, não há estrutura.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import random
import numpy as np
from IPython.display import display

random.seed(42)

# Criar imagem simulando documento escaneado com ruído
largura, altura_img = 620, 480
img = Image.new('RGB', (largura, altura_img), color=(248, 244, 230))  # fundo amarelado de papel velho
draw = ImageDraw.Draw(img)

# Borda simulating scanner
draw.rectangle([15, 15, largura - 15, altura_img - 15], outline=(160, 140, 120), width=2)

# Conteúdo simulando nota fiscal datilografada
linhas_doc = [
    (40,  40,  "NOTA FISCAL Nº 00.123 - 2ª VIA"),
    (40,  75,  "Emitente: COMÉRCIO DE MATERIAIS SÃO PEDRO LTDA"),
    (40,  100, "CNPJ: 12.345.678/0001-99    IE: 123.456.789.000"),
    (40,  125, "End: Rua das Flores, 500 - Centro - São Paulo/SP"),
    (40,  160, "-" * 68),
    (40,  180, "DESCRIÇÃO                    QTD    UNIT      TOTAL"),
    (40,  200, "-" * 68),
    (40,  220, "Cimento CP-II 50kg            10   R$32,00   R$320,00"),
    (40,  240, "Areia média (saco 20kg)        5   R$18,00    R$90,00"),
    (40,  260, "Tijolo cerâmico (cx 50un)      3   R$95,00   R$285,00"),
    (40,  280, "Tinta acrílica branca 18L      2  R$120,00   R$240,00"),
    (40,  300, "-" * 68),
    (40,  325, "TOTAL GERAL:                              R$935,00"),
    (40,  355, "Forma de pgto: DINHEIRO       Data: 28/03/2026"),
    (40,  390, "Assinatura: _____________________________"),
]

for x, y, texto in linhas_doc:
    # Pequena variação na posição para simular imperfeição do scan
    offset_x = random.randint(-1, 1)
    offset_y = random.randint(-1, 1)
    # Cor levemente variável para simular tinta envelhecida
    tom = random.randint(15, 45)
    draw.text((x + offset_x, y + offset_y), texto, fill=(tom, tom, tom + 5))

# Ruído de scanner (pixels aleatórios)
pixels = img.load()
for _ in range(1200):
    x = random.randint(0, largura - 1)
    y = random.randint(0, altura_img - 1)
    v = random.randint(130, 210)
    pixels[x, y] = (v, v - 5, v - 10)

# Mancha leve simulando dobra no papel
for x in range(largura):
    for y in range(410, 440):
        r, g, b = pixels[x, y]
        pixels[x, y] = (min(255, r + 8), min(255, g + 5), min(255, b + 2))

img.save('/content/nota_fiscal_scan.png', dpi=(150, 150))

print("=" * 60)
print("DOCUMENTO ESCANEADO — o que o computador recebe")
print("=" * 60)
display(img)
print(f"\nDimensões: {img.size[0]} x {img.size[1]} pixels")
print(f"Modo: {img.mode}  |  Total de pixels: {img.size[0] * img.size[1]:,}")
print("\nNão há texto. Não há tabela. Não há estrutura.")
print("Só existem valores RGB para cada pixel — é tudo que o computador vê.")

In [ ]:
# Mostrar os valores brutos de pixels da região com texto
import numpy as np

img_array = np.array(img)

print("=" * 60)
print("PIXELS BRUTOS — região onde está o texto 'NOTA FISCAL'")
print("(linhas 35-55, colunas 35-200 da imagem)")
print("=" * 60)
recorte = img_array[35:55, 35:200]
print(f"Shape do recorte: {recorte.shape}  (altura x largura x canais RGB)\n")
print("Valores RGB de cada pixel:")
print(recorte[:5])  # primeiras 5 linhas de pixels
print("\n... Para extrair texto daqui, você precisa de OCR.")
print("Para entender a tabela, você precisa de OCR + detecção de layout.")

---
## Resumo da Parte 1: O Problema

Acabamos de ver o que os dados do mundo real parecem **antes de qualquer tratamento**:

| Tipo | O que vimos | Desafio principal |
|------|-------------|-------------------|
| **HTML** | Tags, atributos, CSS, comentários misturados ao conteúdo | Separar texto relevante do markup |
| **PDF nativo** | Stream binário com operadores gráficos | Reconstruir texto na ordem correta |
| **PDF com tabelas** | Texto posicionado por coordenadas X,Y | Inferir estrutura de grade sem metadados |
| **Documento escaneado** | Matriz de pixels RGB | Não há texto — precisa ser "lido" por OCR |

### Próximo passo: as ferramentas de parsing

Na **Parte 2** vamos resolver cada um desses problemas:  
`BeautifulSoup` → `PyMuPDF / pdfplumber` → `Camelot / tabula` → `Tesseract / EasyOCR`

> *A qualidade do seu pipeline de IA começa aqui — antes do embedding, antes do modelo.*